# **RendaFlex AI**
Assistente Financeiro para pessoas com renda variável.

**O objetivo do projeto** é Classificar transações, Avaliar perfil financeiro e simular decisões.

O dataset utilizado é Synthetic Bank Transactions, do kaggle.com, contendo:

* Clientes – informações básicas sobre os usuários do banco.
* Categorias – categorias padrão de transações utilizadas por muitos bancos em todo o mundo.
* Transações – o núcleo do nosso conjunto de dados, contendo informações básicas sobre as transações, como a conta da contraparte (segunda conta envolvida na transação), a categoria, o valor, entre outros detalhes.
* Assinaturas – informações sobre assinaturas, ou seja, transações realizadas automaticamente.

Além dessas 4 tabelas, usamos também um quinto arquivo, `perfil_financeiro_sintetico.csv`, gerado por
nós. O motivo está explicado na seção 8 — resumindo, o dataset original tem despesa muito maior que
renda pra quase todo cliente (o que não é realista), então a parte de renda/perfil financeiro passou a
vir de uma simulação documentada, enquanto o dataset original continua sendo usado para o que ele faz
bem: classificação de categoria e padrão de consumo.

## 1. Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import time
import json
import joblib

!pip install deep-translator --quiet
from deep_translator import GoogleTranslator

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)


## 2. Carregando os dados

Cinco arquivos: os 4 originais do Synthetic Bank Transactions, mais o `perfil_financeiro_sintetico.csv`
que geramos à parte pra resolver o problema de escala entre renda e despesa (detalhes na seção 8).

In [ ]:
df_clients = pd.read_csv('clients.csv')
df_categories = pd.read_csv('categories.csv')
df_transactions = pd.read_csv('transactions.csv')
df_subscriptions = pd.read_csv('subscriptions.csv')
df_perfil_sintetico = pd.read_csv('perfil_financeiro_sintetico.csv')

for nome, df in [('clients', df_clients), ('categories', df_categories),
                  ('transactions', df_transactions), ('subscriptions', df_subscriptions),
                  ('perfil_sintetico', df_perfil_sintetico)]:
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")


In [ ]:
display(df_clients.head(3))
display(df_categories.head(3))
display(df_transactions.head(3))
display(df_subscriptions.head(3))
display(df_perfil_sintetico.head(3))


## 3. Traduzindo do russo pro português

As 29 categorias do dataset são uma lista pequena e fixa, então em vez de depender de uma API externa
de tradução (que pode falhar por causa de internet, limite de requisição, ou simplesmente não estar
disponível no ambiente onde o notebook está rodando), traduzimos elas com um dicionário fixo, feito à
mão uma única vez. Isso deixa essa etapa 100% confiável e reproduzível, sem depender de nada externo.

Já os nomes de empresa em `subscriptions.csv` são texto mais variado (não dá pra prever todos de
antemão), então ali sim usamos um tradutor automático como estimativa — com cache pra não repetir a
mesma tradução, e um aviso caso a tradução não esteja disponível no ambiente (nesse caso o nome original
em russo é mantido, o que não trava o pipeline, só deixa aquele campo específico sem tradução).

In [ ]:
# Dicionário fixo com as 29 categorias do dataset (feito uma vez, não depende de internet)
DE_PARA_CATEGORIAS = {
    "Каршеринг": "Carsharing",
    "Супермаркеты": "Supermercados",
    "Такси": "Táxi",
    "Музыка": "Música",
    "Фастфуд": "Fast food",
    "Транспорт": "Transporte",
    "Аптеки": "Farmácias",
    "Кино": "Cinema",
    "Книги": "Livros",
    "Развлечения": "Entretenimento",
    "Красота": "Beleza",
    "Образование": "Educação",
    "Одежда и обувь": "Roupas e calçados",
    "Рестораны": "Restaurantes",
    "Топливо": "Combustível",
    "Животные": "Pet shop",
    "Дом и ремонт": "Casa e reforma",
    "Спорттовары": "Artigos esportivos",
    "Сувениры": "Souvenirs",
    "Фото и видео": "Foto e vídeo",
    "Цветы": "Flores",
    "Аренда авто": "Aluguel de carro",
    "Автоуслуги": "Serviços automotivos",
    "Авиабилеты": "Passagens aéreas",
    "Дьюти-фри": "Duty free",
    "Железнодорожные билеты": "Passagens de trem",
    "Искусство": "Arte",
    "Переводы": "Transferências",
    "Другое": "Outros",
}

df_categories = df_categories.rename(columns={
    'id': 'id_categoria',
    'name': 'categoria',
    'mcc-code': 'codigo_mcc'
})

df_categories['categoria'] = df_categories['categoria'].map(DE_PARA_CATEGORIAS).fillna(df_categories['categoria'])

# a coluna description original é uma explicação mais longa de cada categoria; como ela não é usada
# em nenhum cálculo do notebook (só a coluna 'categoria' é), não entra no dicionário manual
df_categories = df_categories.drop(columns=['description'])

display(df_categories.head(29))


In [ ]:
# Tradutor automático, usado só aqui pra nomes de empresa (texto mais variado que categoria).
# Se o ambiente não tiver acesso à internet, a tradução falha silenciosamente e o nome original
# em russo é mantido -- isso não trava o notebook, só deixa esse campo específico sem tradução.
tradutor = GoogleTranslator(source='ru', target='pt')
_cache_traducao = {}

def traduzir(texto, tentativas=2):
    """Traduz um texto do russo pro português, com cache. Se falhar (ex: sem internet),
    devolve o texto original sem travar o notebook."""
    if pd.isna(texto) or str(texto).strip() == '':
        return texto
    texto = str(texto)
    if texto in _cache_traducao:
        return _cache_traducao[texto]
    for tentativa in range(tentativas):
        try:
            traduzido = tradutor.translate(texto)
            _cache_traducao[texto] = traduzido
            return traduzido
        except Exception:
            time.sleep(1)
    _cache_traducao[texto] = texto
    return texto

def traduzir_coluna(df, coluna):
    """Traduz só os valores únicos de uma coluna e aplica o de-para na coluna inteira."""
    valores_unicos = df[coluna].dropna().unique()
    de_para = {v: traduzir(v) for v in valores_unicos}
    return df[coluna].map(de_para).fillna(df[coluna])


In [ ]:
df_subscriptions['product_company'] = traduzir_coluna(df_subscriptions, 'product_company')

df_subscriptions = df_subscriptions.rename(columns={
    'client_id': 'id_cliente',
    'product_category': 'id_categoria',
    'product_company': 'empresa',
    'amount': 'valor',
    'date_start': 'data_inicio',
    'date_end': 'data_fim'
})

qtd_nao_traduzido = df_subscriptions['empresa'].apply(lambda x: any('\u0400' <= c <= '\u04FF' for c in str(x))).sum()
if qtd_nao_traduzido > 0:
    print(f"Aviso: {qtd_nao_traduzido} nomes de empresa continuam em russo "
          f"(provavelmente sem acesso à internet nesse ambiente). Não afeta o restante do notebook, "
          f"já que 'empresa' é usada só como informação de apoio (seção 5), não em nenhum cálculo.")

display(df_subscriptions.head(10))


## 4. Preparando as transações

Ajustamos nomes de coluna, convertemos a data e trazemos o nome da categoria (já traduzida) pra cada
transação. Esse é o dataset que vamos usar pra classificação de despesa e pra montar exemplos reais de
transação mais adiante — não mais pra calcular renda e saldo (isso agora vem do arquivo sintético).

In [ ]:
df_transactions_clean = df_transactions.rename(columns={
    'client_id': 'id_cliente',
    'product_category': 'id_categoria',
    'product_company': 'empresa',
    'subtype': 'subtipo',
    'amount': 'valor',
    'date': 'data',
    'transaction_type': 'tipo_transacao'
}).drop(columns=[c for c in ['Unnamed: 0'] if c in df_transactions.columns])

df_transactions_clean['data'] = pd.to_datetime(df_transactions_clean['data'])
df_transactions_clean['ano_mes'] = df_transactions_clean['data'].dt.to_period('M')

df_transactions_clean = df_transactions_clean.merge(
    df_categories[['id_categoria', 'categoria', 'codigo_mcc']],
    on='id_categoria', how='left'
)

print(df_transactions_clean['tipo_transacao'].value_counts())
display(df_transactions_clean.head(5))


## 5. Assinaturas como apoio qualitativo

Não usamos mais `subscriptions.csv` pra calcular o valor da despesa fixa (isso já vem simulado no
arquivo de perfil). Mas continua útil pra dar exemplos realistas de "assinatura recorrente" — por
exemplo, pra mostrar no pitch quais empresas aparecem mais como gasto fixo entre os clientes.

In [ ]:
top_assinaturas = df_subscriptions['empresa'].value_counts().head(10)
print("Assinaturas mais comuns no dataset original (uso qualitativo/apoio):")
print(top_assinaturas)


## 6. Organizando os dados demográficos dos clientes

De `clients.csv` ficamos só com o que tem valor pro modelo: idade e gênero. Nome, endereço, telefone,
e-mail, local de trabalho e os campos financeiros (`income`, `expenses`, `credit`, `deposit`) saem —
os financeiros porque são justamente os campos que geravam a distorção original entre renda e despesa,
substituídos agora pelo arquivo sintético.

In [ ]:
df_clients_clean = df_clients.rename(columns={'id': 'id_cliente', 'birthdate': 'data_nascimento', 'gender': 'genero'})
df_clients_clean['data_nascimento'] = pd.to_datetime(df_clients_clean['data_nascimento'], errors='coerce')
df_clients_clean['idade'] = (pd.Timestamp('2020-12-31') - df_clients_clean['data_nascimento']).dt.days // 365
df_clients_clean = df_clients_clean[['id_cliente', 'idade', 'genero']]

display(df_clients_clean.head(5))


## 7. Por que criamos um dataset sintético de renda e perfil

Ao calcular renda e despesa reais a partir de `transactions.csv`, a despesa média mensal por cliente
saiu ~14x maior que a renda média mensal — o que não é realista e fazia o modelo classificar 100% dos
clientes como "Em risco". Investigando, a causa é que o dataset tem muito mais transações de despesa
(cartão, compras do dia a dia) do que de renda (poucos depósitos grandes por ano) — ele parece simular
só o fluxo de pagamentos, não a renda completa da pessoa.

Decisão tomada com o time: manter o `transactions.csv` só para classificação de categoria e padrão de
consumo (onde ele funciona bem), e gerar separadamente um dataset simulado — documentado e reprodutível
— só pra parte de renda/perfil financeiro. Esse é o `perfil_financeiro_sintetico.csv`.

Resumo de como ele foi gerado (script à parte, com seed fixa = 42, reprodutível):
- Cada cliente recebe um perfil de renda sorteado: Estável (35%), Moderadamente variável (40%) ou
  Altamente variável (25%) — pesando mais pro público-alvo de renda variável do projeto.
- A renda-base mensal segue distribuição log-normal (média ~R\$3.800), e a instabilidade de renda
  (coeficiente de variação) reflete o perfil sorteado.
- A despesa é sorteada como fração da própria renda do cliente (mantendo despesa e renda na mesma
  escala, ao contrário do dataset original).
- A despesa fixa é uma fração da despesa total.
- O perfil final (Saudável / Em observação / Em risco) usa a mesma regra de negócio do notebook,
  aplicada em cima desses números simulados.

In [ ]:
features_cliente = df_perfil_sintetico.merge(df_clients_clean[['id_cliente']], on='id_cliente', how='inner')
# idade/gênero já vêm dentro do próprio arquivo sintético (foram herdados do clients.csv na geração)

print(features_cliente['perfil_financeiro'].value_counts())
display(features_cliente.head(10))


## 8. Conferindo se a base sintética faz sentido

Checagem visual rápida: instabilidade de renda e comprometimento com despesa fixa devem crescer conforme
o perfil piora (Saudável -> Em observação -> Em risco). Se os grupos aparecessem todos misturados, seria
sinal de que algo na simulação precisa de ajuste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=features_cliente, x='perfil_financeiro', y='coef_variacao_renda',
            order=['Saudável', 'Em observação', 'Em risco'], ax=axes[0])
axes[0].set_title('Instabilidade de renda por perfil')

sns.boxplot(data=features_cliente, x='perfil_financeiro', y='comprometimento_fixo',
            order=['Saudável', 'Em observação', 'Em risco'], ax=axes[1])
axes[1].set_title('Comprometimento com despesa fixa por perfil')

plt.tight_layout()
plt.show()


## 9. Classificando a descrição da transação em categoria

Aqui voltamos a usar o `transactions.csv` — a parte que ele resolve bem. O objetivo é treinar um modelo
que, dado um texto de descrição de transação (ex: "Supermercado", "Netflix"), preveja em qual categoria
do hackathon ela se encaixa (Alimentação, Transporte, Saúde, Moradia, Educação, Lazer, Serviços, Outras).

Como o dataset não tem descrição livre de transação (só categoria numérica), montamos aqui uma base de
exemplos descrição -> categoria pra treinar esse classificador. Isso é um ponto de partida pra provar o
pipeline funcionando; o ideal é o time trocar/ampliar essa base por uma fonte real de descrições assim
que definirem qual usar.

In [ ]:
exemplos_descricao = pd.DataFrame([
    ("Supermercado Extra", "Alimentação"),
    ("iFood pedido restaurante", "Alimentação"),
    ("Feira livre", "Alimentação"),
    ("Padaria do bairro", "Alimentação"),
    ("Posto de gasolina", "Transporte"),
    ("Uber viagem centro", "Transporte"),
    ("Passagem de ônibus", "Transporte"),
    ("Estacionamento shopping", "Transporte"),
    ("Farmácia remédio", "Saúde"),
    ("Consulta médica particular", "Saúde"),
    ("Plano de saúde mensalidade", "Saúde"),
    ("Academia mensalidade", "Saúde"),
    ("Aluguel apartamento", "Moradia"),
    ("Conta de luz", "Moradia"),
    ("Conta de água", "Moradia"),
    ("Condomínio mensal", "Moradia"),
    ("Mensalidade faculdade", "Educação"),
    ("Curso online plataforma", "Educação"),
    ("Material escolar", "Educação"),
    ("Livro técnico", "Educação"),
    ("Netflix assinatura", "Lazer"),
    ("Cinema ingresso", "Lazer"),
    ("Show de música", "Lazer"),
    ("Spotify assinatura", "Lazer"),
    ("Salão de beleza", "Serviços"),
    ("Conserto celular", "Serviços"),
    ("Assinatura software", "Serviços"),
    ("Plano de celular", "Serviços"),
    ("Transferência entre contas", "Outras"),
    ("Saque em caixa eletrônico", "Outras"),
], columns=["descricao", "categoria"])

display(exemplos_descricao)


In [ ]:
X_texto = exemplos_descricao['descricao']
y_texto = exemplos_descricao['categoria']

X_texto_treino, X_texto_teste, y_texto_treino, y_texto_teste = train_test_split(
    X_texto, y_texto, test_size=0.25, random_state=42, stratify=y_texto
)

vetorizador_tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
X_texto_treino_vet = vetorizador_tfidf.fit_transform(X_texto_treino)
X_texto_teste_vet = vetorizador_tfidf.transform(X_texto_teste)

modelo_categoria = MultinomialNB()
modelo_categoria.fit(X_texto_treino_vet, y_texto_treino)

y_texto_pred = modelo_categoria.predict(X_texto_teste_vet)

print("Acurácia:", accuracy_score(y_texto_teste, y_texto_pred))
print()
print(classification_report(y_texto_teste, y_texto_pred, zero_division=0))


## 10. Treinando o modelo de perfil financeiro

Usamos as colunas do dataset sintético (renda média, instabilidade de renda, comprometimento com
despesa fixa, despesa média, saldo médio, idade) pra prever o perfil financeiro. RandomForest porque
lida bem com dado tabular, treina rápido e dá pra explicar quais variáveis pesaram mais na decisão.

In [ ]:
colunas_features = [
    'renda_media', 'coef_variacao_renda', 'despesa_media_mensal',
    'comprometimento_fixo', 'saldo_medio_mensal', 'idade'
]

base_modelo = features_cliente.dropna(subset=colunas_features + ['perfil_financeiro']).copy()

X_perfil = base_modelo[colunas_features]
y_perfil = base_modelo['perfil_financeiro']

X_perfil_treino, X_perfil_teste, y_perfil_treino, y_perfil_teste = train_test_split(
    X_perfil, y_perfil, test_size=0.2, random_state=42, stratify=y_perfil
)

modelo_perfil = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
modelo_perfil.fit(X_perfil_treino, y_perfil_treino)

y_perfil_pred = modelo_perfil.predict(X_perfil_teste)

print("Acurácia:", accuracy_score(y_perfil_teste, y_perfil_pred))
print("F1 (média ponderada):", f1_score(y_perfil_teste, y_perfil_pred, average='weighted'))
print()
print(classification_report(y_perfil_teste, y_perfil_pred, zero_division=0))


In [ ]:
matriz = confusion_matrix(y_perfil_teste, y_perfil_pred, labels=modelo_perfil.classes_)

plt.figure(figsize=(6, 5))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=modelo_perfil.classes_, yticklabels=modelo_perfil.classes_)
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.title('Matriz de confusão - perfil financeiro')
plt.show()

importancias = pd.Series(modelo_perfil.feature_importances_, index=colunas_features).sort_values(ascending=False)
print(importancias)


## 11. Salvando os modelos treinados

Serializamos os dois modelos (categoria de despesa e perfil financeiro) junto com o vetorizador TF-IDF.
É esse arquivo que o back-end vai carregar pra não precisar retreinar nada a cada chamada da API.

In [ ]:
joblib.dump(modelo_categoria, 'modelo_categoria_despesa.pkl')
joblib.dump(vetorizador_tfidf, 'vetorizador_tfidf_categoria.pkl')
joblib.dump(modelo_perfil, 'modelo_perfil_financeiro.pkl')

features_cliente.to_csv('features_cliente_rendaflex.csv', index=False)
df_categories.to_csv('categorias_traduzidas.csv', index=False)

print("Modelos e tabelas salvos com sucesso.")


## 12. Função de inferência (o que a API vai chamar)

Recebe os dados financeiros do usuário no formato do edital, classifica cada transação numa categoria,
calcula o perfil financeiro e devolve recomendações. É essa função (ou a lógica dela) que o back-end
empacota dentro da API.

In [ ]:
def analisar_financas(dados_entrada):
    """
    Recebe um dicionário no formato do endpoint POST /analise-financeira e devolve
    o resumo de gastos, o perfil financeiro e recomendações, em formato de dicionário
    (pronto pra virar JSON).
    """
    transacoes = dados_entrada.get('transacoes', [])
    renda_mensal = dados_entrada.get('renda_mensal', 0)
    nivel_endividamento = dados_entrada.get('nivel_endividamento', 0)
    frequencia_poupanca = dados_entrada.get('frequencia_poupanca', 'Baixa')

    descricoes = [t['descricao'] for t in transacoes]
    valores = [t['valor'] for t in transacoes]

    if descricoes:
        vetor = vetorizador_tfidf.transform(descricoes)
        categorias_previstas = modelo_categoria.predict(vetor)
    else:
        categorias_previstas = []

    resumo_gastos = {}
    for categoria, valor in zip(categorias_previstas, valores):
        chave = categoria.lower().replace('ç', 'c').replace('ã', 'a')
        resumo_gastos[chave] = resumo_gastos.get(chave, 0) + valor

    despesa_total = sum(valores)

    mapa_frequencia = {'Alta': 0.1, 'Media': 0.3, 'Baixa': 0.6}
    coef_variacao_aprox = mapa_frequencia.get(frequencia_poupanca, 0.3)

    comprometimento = despesa_total / renda_mensal if renda_mensal > 0 else 1
    saldo_estimado = renda_mensal - despesa_total

    entrada_modelo = pd.DataFrame([{
        'renda_media': renda_mensal,
        'coef_variacao_renda': coef_variacao_aprox,
        'despesa_media_mensal': despesa_total,
        'comprometimento_fixo': min(comprometimento, 1),
        'saldo_medio_mensal': saldo_estimado,
        'idade': dados_entrada.get('idade', 35)
    }])

    perfil_previsto = modelo_perfil.predict(entrada_modelo)[0]
    probabilidade = modelo_perfil.predict_proba(entrada_modelo).max()

    recomendacoes = []
    if nivel_endividamento > 30:
        recomendacoes.append("Priorizar redução do nível de endividamento")
    if comprometimento > 0.6:
        recomendacoes.append("Rever gastos recorrentes, o comprometimento da renda está alto")
    if frequencia_poupanca == 'Baixa':
        recomendacoes.append("Aumentar a frequência de poupança mensal")
    if not recomendacoes:
        recomendacoes.append("Manter os hábitos financeiros atuais")

    return {
        "perfil_financeiro": perfil_previsto,
        "probabilidade": round(float(probabilidade), 2),
        "resumo_gastos": resumo_gastos,
        "recomendacoes": recomendacoes
    }


## 13. Simulando o impacto de uma nova despesa

Essa é a funcionalidade do MVP que faltava: dado o resultado de `analisar_financas` pra um usuário, simular
"e se essa pessoa assumisse mais uma despesa de R\$X por mês, em tal categoria?" — recalculando o
comprometimento e rodando de novo o modelo de perfil, sem precisar coletar tudo de novo. É basicamente a
mesma função de análise, mas com uma transação hipotética somada às demais.

In [ ]:
def simular_nova_despesa(dados_entrada, descricao_nova_despesa, valor_nova_despesa):
    """
    Recebe a mesma entrada de analisar_financas, mais a descrição e o valor de uma despesa
    hipotética, e devolve a comparação entre o perfil financeiro atual e o perfil financeiro
    projetado caso essa despesa nova seja assumida.
    """
    resultado_atual = analisar_financas(dados_entrada)

    dados_com_nova_despesa = dict(dados_entrada)
    dados_com_nova_despesa['transacoes'] = list(dados_entrada.get('transacoes', [])) + [
        {"descricao": descricao_nova_despesa, "valor": valor_nova_despesa}
    ]

    resultado_projetado = analisar_financas(dados_com_nova_despesa)

    return {
        "perfil_atual": resultado_atual['perfil_financeiro'],
        "perfil_projetado": resultado_projetado['perfil_financeiro'],
        "houve_piora": resultado_atual['perfil_financeiro'] != resultado_projetado['perfil_financeiro'],
        "resumo_gastos_projetado": resultado_projetado['resumo_gastos'],
        "recomendacoes_projetadas": resultado_projetado['recomendacoes']
    }


## 14. Montando 3 exemplos reais (um de cada perfil)

Pegamos 1 cliente de verdade de cada perfil (Saudável, Em observação, Em risco) da tabela
`features_cliente` — que agora vem do dataset sintético de renda — e reconstruímos a entrada da API
com uma amostra de despesas reais desse cliente, tiradas de `transactions.csv`. Assim renda e perfil
vêm da simulação documentada, mas as transações de exemplo continuam sendo dado real do dataset original.

In [ ]:
def montar_entrada_real(id_cliente, n_transacoes=5):
    """Monta a entrada da API pra um cliente: renda/perfil vêm do dataset sintético,
    e a amostra de transações vem do histórico real dele em transactions.csv."""

    linha = features_cliente[features_cliente['id_cliente'] == id_cliente].iloc[0]

    despesas_cliente = df_transactions_clean[
        (df_transactions_clean['id_cliente'] == id_cliente) &
        (df_transactions_clean['tipo_transacao'] == 'Negative')
    ].sort_values('data', ascending=False)

    if despesas_cliente.empty:
        transacoes = []
    else:
        ultimo_mes = despesas_cliente['ano_mes'].iloc[0]
        amostra = despesas_cliente[despesas_cliente['ano_mes'] == ultimo_mes].head(n_transacoes)
        transacoes = [
            {"descricao": row['categoria'], "valor": round(float(row['valor']), 2)}
            for _, row in amostra.iterrows()
        ]

    if linha['coef_variacao_renda'] > 0.4:
        frequencia_poupanca = "Baixa"
    elif linha['coef_variacao_renda'] > 0.2:
        frequencia_poupanca = "Media"
    else:
        frequencia_poupanca = "Alta"

    return {
        "renda_mensal": round(float(linha['renda_media']), 2),
        "nivel_endividamento": round(float(linha['comprometimento_fixo']) * 100, 1),
        "frequencia_poupanca": frequencia_poupanca,
        "transacoes": transacoes,
        "idade": int(linha['idade']) if not pd.isna(linha['idade']) else 35
    }


In [ ]:
clientes_exemplo = {}
for perfil in ['Saudável', 'Em observação', 'Em risco']:
    candidatos = features_cliente[features_cliente['perfil_financeiro'] == perfil]
    if not candidatos.empty:
        clientes_exemplo[perfil] = candidatos.iloc[0]['id_cliente']

print("Clientes escolhidos:", clientes_exemplo)

exemplos_reais = {}
for perfil, id_cliente in clientes_exemplo.items():
    exemplos_reais[perfil] = montar_entrada_real(id_cliente)

for perfil, entrada in exemplos_reais.items():
    print(f"--- Exemplo real ({perfil}, cliente {clientes_exemplo[perfil]}) ---")
    print("Entrada:", entrada)
    resultado = analisar_financas(entrada)
    print("Saída:", resultado)
    print()


## 15. Testando a simulação de nova despesa

Usamos o exemplo "Saudável" pra mostrar o caso de uso central do produto: simular se uma pessoa aguenta
assumir uma despesa nova sem virar "Em risco".

In [ ]:
if 'Saudável' in exemplos_reais:
    entrada_teste = exemplos_reais['Saudável']
    simulacao = simular_nova_despesa(entrada_teste, "Financiamento de carro", 900)
    print("Simulação de nova despesa (cliente Saudável + financiamento de R$900):")
    print(simulacao)


## 16. Exportando o contrato de entrada/saída pra Back-end e Front-end

Salvamos os 3 exemplos (entrada + saída da análise + saída da simulação de nova despesa) num `.json`
separado do notebook — esse arquivo é o que dá pra mandar direto pros times de Back e Front, sem
precisar abrir Python pra entender o formato.

In [ ]:
contrato_exemplos = []
for perfil, entrada in exemplos_reais.items():
    saida = analisar_financas(entrada)
    simulacao = simular_nova_despesa(entrada, "Nova assinatura de streaming", 45)
    contrato_exemplos.append({
        "perfil_origem": perfil,
        "id_cliente_origem": int(clientes_exemplo[perfil]),
        "entrada": entrada,
        "saida_analise": saida,
        "exemplo_simulacao_nova_despesa": simulacao
    })

with open('contrato_api_exemplos.json', 'w', encoding='utf-8') as f:
    json.dump(contrato_exemplos, f, ensure_ascii=False, indent=2)

print("Contrato salvo em contrato_api_exemplos.json")


## 17. Próximos passos

- Validar com o time se a metodologia do dataset sintético (seção 7) é aceitável pra defender no
  hackathon, ou se querem ajustar algum parâmetro da simulação.
- Trocar a base de exemplos de descrição da seção 9 por uma base maior e real, assim que o time
  decidir qual fonte usar.
- Entregar pro back-end: os arquivos `.pkl` da seção 11, as funções `analisar_financas` e
  `simular_nova_despesa` da seção 12-13 (viram um `inference.py`), e o `contrato_api_exemplos.json`.
- Entregar pro front-end: apenas o `contrato_api_exemplos.json` — não precisam dos CSVs brutos nem
  do notebook.
- Ainda em aberto no MVP e fora do escopo deste notebook: dashboard simples e integração com OCI —
  ambos ficam a cargo do time de Back/Front na próxima fase, conforme distribuição combinada em reunião.
